In [ ]:
#0a BOE download
import io, os, zipfile, requests
import pandas as pd

BASE = "https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/"
URLS = {"daily_archive": BASE + "glcnominalddata.zip",
        "latest": BASE + "latest-yield-curve-data.zip"}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

# 1. Download and unzip
for name, url in URLS.items():
    os.makedirs(f"boe/{name}", exist_ok=True)
    try:
        r = requests.get(url, headers=HEADERS, timeout=180)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall(f"boe/{name}")
        print(f"{name}: downloaded {len(r.content)/1e6:.1f} MB")
    except Exception as e:
        print(f"{name}: download failed ({e})")

is_xl = lambda f: f.lower().endswith((".xlsx", ".xls"))

# 2. What is inside
for name in URLS:
    folder = f"boe/{name}"
    if not os.path.isdir(folder):
        continue
    print("\n" + "=" * 70 + f"\n{name}")
    for f in sorted(x for x in os.listdir(folder) if is_xl(x)):
        try:
            xl = pd.ExcelFile(f"{folder}/{f}")
            print(f"\n{f}\n  sheets: {xl.sheet_names}")
        except Exception as e:
            print(f"\n{f}: could not open ({e})")

# 3. Peek at the spot-rate sheets (first and last archive workbook, and the latest nominal workbook)
def peek(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    cols = sorted(set(list(range(min(4, df.shape[1]))) + [df.shape[1] - 2, df.shape[1] - 1]))
    print(f"\n{os.path.basename(path)} | {sheet} | shape {df.shape}")
    print(df.iloc[:7, cols].to_string(header=False))
    print("   ...")
    print(df.iloc[-2:, cols].to_string(header=False))
    dates = pd.to_datetime(df.iloc[:, 0], errors="coerce").dropna()
    if len(dates):
        print(f"   dates in column 1: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d} ({len(dates):,} rows)")

arch = sorted(f for f in os.listdir("boe/daily_archive") if is_xl(f)) if os.path.isdir("boe/daily_archive") else []
late = [f for f in sorted(os.listdir("boe/latest")) if is_xl(f) and "glc" in f.lower() and "nominal" in f.lower()] \
       if os.path.isdir("boe/latest") else []
for folder, fl in [("daily_archive", arch[:1] + arch[-1:]), ("latest", late)]:
    for f in fl:
        xl = pd.ExcelFile(f"boe/{folder}/{f}")
        for s in [s for s in xl.sheet_names if "spot" in s.lower()] or xl.sheet_names[:2]:
            peek(f"boe/{folder}/{f}", s)

In [ ]:
#0b BOE stitch
import os
from datetime import datetime

is_xl = lambda f: f.lower().endswith((".xlsx", ".xls"))

def load(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    yrs = pd.to_numeric(df.iloc[3, 1:], errors="coerce").round(4)      # 'years:' row
    body = df.iloc[4:]
    keep = body.iloc[:, 0].map(lambda v: isinstance(v, datetime))
    out = body.loc[keep].iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
    out.columns = yrs.values
    out.index = pd.to_datetime(body.loc[keep].iloc[:, 0])
    return out.loc[:, ~pd.isna(out.columns)]

def stitch(short):
    parts = []
    files = [("boe/daily_archive", f) for f in sorted(os.listdir("boe/daily_archive")) if is_xl(f)]
    files += [("boe/latest", "GLC Nominal daily data current month.xlsx")]   # last, so it wins on duplicates
    for folder, f in files:
        xl = pd.ExcelFile(f"{folder}/{f}")
        s = next(x for x in xl.sheet_names if "spot" in x and ("short" in x) == short)
        parts.append(load(f"{folder}/{f}", s))
    d = pd.concat(parts).sort_index()
    return d[~d.index.duplicated(keep="last")].dropna(how="all")

spot, short = stitch(False), stitch(True)       # takes a minute or two
spot.to_csv("gilt_spot_curve.csv"); short.to_csv("gilt_spot_short_end.csv")

print("Whole curve:", spot.shape, spot.index.min().date(), "to", spot.index.max().date())
def cover(d, mats):
    mats = [c for c in mats if c in d.columns]
    f = lambda c: d[c].loc[d[c].first_valid_index():].notna().mean() * 100
    return pd.DataFrame({"first valid": [d[c].first_valid_index().date() for c in mats],
                         "% filled after start": [round(f(c)) for c in mats],
                         "% filled since 2020": [round(d[c].loc["2020":].notna().mean() * 100) for c in mats]},
                        index=mats)
print(cover(spot, [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40]))
print("\nShort end (maturity in years):")
print(cover(short, [round(m / 12, 4) for m in (1, 2, 3, 6, 12, 24, 60)]))
print("\nLowest yield: %.2f | negative cells: %d" % (spot.min().min(), (spot < 0).sum().sum()))


In [ ]:
#0c zero bootstrap loads the FED curve
!pip install fredapi -q
import io, requests, numpy as np, pandas as pd
from fredapi import Fred
from google.colab import userdata

fred = Fred(api_key=userdata.get("FRED_API_KEY"))
ser = {"DGS1MO": 1/12, "DGS3MO": .25, "DGS6MO": .5, "DGS1": 1, "DGS2": 2, "DGS3": 3,
       "DGS5": 5, "DGS7": 7, "DGS10": 10, "DGS20": 20, "DGS30": 30}
Y = pd.DataFrame({s: fred.get_series(s, observation_start="2001-07-31") for s in ser}).dropna()
mats = np.array(list(ser.values()))

# 1. Par yields -> zero-coupon yields (bootstrap on a semiannual grid; continuously compounded, %)
grid = np.arange(1, 61) * 0.5
P = np.array([np.interp(grid, mats[2:], r[2:]) for r in Y.values / 100])
d, ann = np.zeros_like(P), np.zeros(len(P))
for k in range(60):
    c = P[:, k] / 2
    d[:, k] = (1 - c * ann) / (1 + c)
    ann += d[:, k]
Zg = -np.log(d) / grid * 100
Zb = np.log1p(Y.values[:, :2] / 100 * mats[:2]) / mats[:2] * 100                 # 1M and 3M bills
Zc = np.column_stack([Zb, Zg[:, (mats[2:] / 0.5 - 1).astype(int)]])               # zeros at the 11 CMT maturities

# 2. NSS with fixed lambdas, betas by OLS each day
def basis(tau, l1, l2):
    x1, x2 = tau / l1, tau / l2
    f1 = (1 - np.exp(-x1)) / x1
    return np.column_stack([np.ones_like(tau), f1, f1 - np.exp(-x1), (1 - np.exp(-x2)) / x2 - np.exp(-x2)])

fits = {}
for l1 in [0.3, 0.5, 0.75, 1, 1.5, 2, 3, 4]:
    for l2 in [4, 6, 8, 10, 13, 16, 20]:
        X = basis(mats, l1, l2)
        B = Zc @ np.linalg.pinv(X).T
        fits[(l1, l2)] = (B, ((Zc - B @ X.T) ** 2).sum(1))
pooled = min(fits, key=lambda k: fits[k][1].sum())
B, sse = fits[pooled]
free = np.min([v[1] for v in fits.values()], axis=0)                             # lambdas re-chosen every day
rm = lambda s: np.sqrt(s / len(mats)) * 100                                      # bp
print("Fixed lambdas: l1=%g, l2=%g | RMSE mean %.1f bp, max %.1f bp, days >5bp: %d" % (
    *pooled, rm(sse).mean(), rm(sse).max(), (rm(sse) > 5).sum()))
print("Free lambdas each day (grid): RMSE mean %.1f bp (the cost of fixing them)" % rm(free).mean())

params_z = pd.DataFrame(B, index=Y.index, columns=["b0", "b1", "b2", "b3"])
params_z["l1"], params_z["l2"], params_z["rmse"] = pooled[0], pooled[1], np.sqrt(sse / len(mats))
params_z.to_csv("nss_params_zero.csv")
pd.DataFrame(Zc, index=Y.index, columns=Y.columns).to_csv("zero_yields.csv")

# 3. Benchmark against the Fed's GSW zero-coupon curve
UA = {"User-Agent": "Mozilla/5.0"}
CSV_URL = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
XLS_URL = "https://www.federalreserve.gov/pubs/feds/2006/200628/feds200628.xls"

def load_gsw():
    for url in (CSV_URL, XLS_URL):
        try:
            r = requests.get(url, headers=UA, timeout=300); r.raise_for_status()
            if r.content[:2] == b"\xd0\xcf":                                       # old .xls format
                raw = pd.read_excel(io.BytesIO(r.content), header=None)
                h = next(i for i in range(40) if raw.iloc[i].astype(str).str.upper().str.contains("SVENY01").any())
                df = raw.iloc[h + 1:].copy(); df.columns = raw.iloc[h]
            else:
                lines = r.text.splitlines()
                h = next(i for i, l in enumerate(lines[:40]) if "SVENY01" in l.upper())
                df = pd.read_csv(io.StringIO("\n".join(lines[h:])))
            df.columns = [str(c).strip().upper() for c in df.columns]
            s = df.iloc[:, 0].astype(str).str.replace(" ", "", regex=False)
            df.index = pd.to_datetime(s, format="%Y:%m:%d", errors="coerce").fillna(pd.to_datetime(s, errors="coerce"))
            df = df.loc[df.index.notna()].drop(columns=df.columns[0]).apply(pd.to_numeric, errors="coerce")
            print("GSW loaded from", url.split("/")[-1])
            return df.sort_index().loc["2001-07-31":]
        except Exception as e:
            print("GSW load failed:", url.split("/")[-1], "|", str(e)[:100])

def nss_old(t, Q):
    t = np.asarray(t, float)[None, :]; g = lambda c: Q[c].values[:, None]
    x1, x2 = t / g("l1"), t / g("l2"); f1 = (1 - np.exp(-x1)) / x1
    return g("b0") + g("b1") * f1 + g("b2") * (f1 - np.exp(-x1)) + g("b3") * ((1 - np.exp(-x2)) / x2 - np.exp(-x2))

G = load_gsw()
if G is not None:
    tg = np.array([1, 2, 3, 5, 7, 10, 20, 30], float)
    Gz = G[[f"SVENY{int(t):02d}" for t in tg]].reindex(Y.index).values
    def gap(F):
        e = (F - Gz) * 100
        return np.nanmean(e, 0), np.sqrt(np.nanmean(e ** 2, 0))
    res = {}
    try:
        old = pd.read_csv("nss_params.csv", index_col=0, parse_dates=True).reindex(Y.index)
        res["old par-based: bias bp"], res["old par-based: RMSE bp"] = gap(nss_old(tg, old))
    except Exception as e:
        print("Old fit not compared:", str(e)[:80])
    res["new zero fit: bias bp"], res["new zero fit: RMSE bp"] = gap(params_z[["b0", "b1", "b2", "b3"]].values @ basis(tg, *pooled).T)
    tab = pd.DataFrame(res); tab.index = [f"{int(t)}Y" for t in tg]
    print("\nFitted curve minus Fed GSW zero-coupon yield (%d common days):" % np.isfinite(Gz[:, 0]).sum())
    print(tab.round(1).to_string())

In [ ]:
#0d diagnostic, but #0e needs the variables it defines
tg = [1, 2, 3, 5, 7, 10, 20, 30]
rep = lambda E: (np.nanmean(E, 0), np.sqrt(np.nanmean(E ** 2, 0)))
gz = G[[f"SVENY{t:02d}" for t in tg]].reindex(Y.index).values          # Fed zero-coupon yields
gp = G[[f"SVENPY{t:02d}" for t in tg]].reindex(Y.index).values         # Fed par yields
par = Y[["DGS1", "DGS2", "DGS3", "DGS5", "DGS7", "DGS10", "DGS20", "DGS30"]].values
a, b, c = rep((par - gp) * 100), rep((Zc[:, 3:] - gz) * 100), rep((gz - gp) * 100)
print(pd.DataFrame({"CMT par - Fed par: bias": a[0], "CMT par - Fed par: RMSE": a[1],
                    "my bootstrapped zero - Fed zero: bias": b[0], "my bootstrapped zero - Fed zero: RMSE": b[1],
                    "Fed zero - Fed par (size of issue 1)": c[0]},
                   index=[f"{t}Y" for t in tg]).round(1).to_string())

resid = (Zc - B @ basis(mats, *pooled).T) * 100
print("\nFixed-lambda NSS residual vs my bootstrapped zero, RMSE bp:")
print(pd.Series(np.sqrt((resid ** 2).mean(0)), index=["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]).round(1).to_string())

In [ ]:
#0e final fit, saves nss_params_zero.csv and zero_yields.csv
from scipy.interpolate import PchipInterpolator
L1, L2 = [0.3, 0.5, 0.75, 1, 1.5, 2, 3, 4], [4, 6, 8, 10, 13, 16, 20]

def zeros(v):
    P = []
    for r in Y.values / 100:
        x, y = mats[2:], r[2:]
        if v == "no20": x, y = np.delete(x, 7), np.delete(y, 7)
        P.append(np.interp(grid, x, y) if v == "linear" else PchipInterpolator(x, y)(grid))
    P = np.array(P); d, ann = np.zeros_like(P), np.zeros(len(P))
    for k in range(60):
        c = P[:, k] / 2
        d[:, k] = (1 - c * ann) / (1 + c); ann += d[:, k]
    return np.column_stack([Zb, (-np.log(d) / grid * 100)[:, (mats[2:] / 0.5 - 1).astype(int)]])

def fit_smooth(Z):
    keys = [(a, b) for a in L1 for b in L2]
    sse = np.column_stack([((Z - Z @ np.linalg.pinv(basis(mats, *k)).T @ basis(mats, *k).T) ** 2).sum(1) for k in keys])
    bk = sse.argmin(1)
    sm = lambda x: pd.Series(x, index=Y.index).rolling(21, min_periods=1, center=True).median().values
    l1s, l2s = sm([keys[i][0] for i in bk]), sm([keys[i][1] for i in bk])
    Bs, s2, F = np.empty((len(Y), 4)), np.empty(len(Y)), np.empty((len(Y), len(tg)))
    for i in range(len(Y)):
        X = basis(mats, l1s[i], l2s[i])
        Bs[i] = np.linalg.lstsq(X, Z[i], rcond=None)[0]
        s2[i] = ((Z[i] - X @ Bs[i]) ** 2).sum()
        F[i] = basis(np.array(tg, float), l1s[i], l2s[i]) @ Bs[i]
    return Bs, l1s, l2s, s2, F

Zs = {v: zeros(v) for v in ("linear", "pchip", "no20")}
ev = lambda Z: np.sqrt(np.nanmean(((Z[:, [8, 9, 10]] - gz[:, [5, 6, 7]]) * 100) ** 2, 0))
print("Bootstrapped zero vs Fed zero, RMSE bp (10Y, 20Y, 30Y):")
for v in Zs: print(" ", v, ev(Zs[v]).round(1))
best = min(Zs, key=lambda v: ev(Zs[v]).sum()); print("Chosen:", best)

Zc = Zs[best]
Bs, l1s, l2s, s2, F = fit_smooth(Zc)
print("Smoothed lambdas: RMSE mean %.1f bp, max %.1f bp, days >5bp: %d" % (rm(s2).mean(), rm(s2).max(), (rm(s2) > 5).sum()))
e = (F - gz) * 100
print(pd.DataFrame({"bias bp": np.nanmean(e, 0), "RMSE bp": np.sqrt(np.nanmean(e ** 2, 0))},
                   index=[f"{t}Y" for t in tg]).round(1).T.to_string())
params_z = pd.DataFrame(Bs, index=Y.index, columns=["b0", "b1", "b2", "b3"])
params_z["l1"], params_z["l2"], params_z["rmse"] = l1s, l2s, np.sqrt(s2 / len(mats))
params_z.to_csv("nss_params_zero.csv"); pd.DataFrame(Zc, index=Y.index, columns=Y.columns).to_csv("zero_yields.csv")

In [ ]:
# 1 RELOAD
!pip install fredapi -q
import numpy as np, pandas as pd
from fredapi import Fred
from google.colab import userdata
fred = Fred(api_key=userdata.get("FRED_API_KEY"))
try: G[[f"SVENY{t:02d}" for t in range(1, 31)]].to_csv("gsw_zero.csv")      # saves the Fed curve if still in memory
except NameError: pass
G = pd.read_csv("gsw_zero.csv", index_col=0, parse_dates=True)
yields = pd.read_csv("zero_yields.csv", index_col=0, parse_dates=True)       # zero yields, same column names as before
params = pd.read_csv("nss_params_zero.csv", index_col=0, parse_dates=True)
cols = ["b0", "b1", "b2", "b3", "l1", "l2"]
mats = np.array([1/12, .25, .5, 1, 2, 3, 5, 7, 10, 20, 30]); series = dict(zip(yields.columns, mats))
lower = np.array([-np.inf] * 4 + [0.3, 4.0]); upper = np.array([np.inf] * 4 + [4.0, 20.0])
def nss(tau, b0, b1, b2, b3, l1, l2):
    t1, t2 = tau / l1, tau / l2
    f1 = (1 - np.exp(-t1)) / t1
    return b0 + b1 * f1 + b2 * (f1 - np.exp(-t1)) + b3 * ((1 - np.exp(-t2)) / t2 - np.exp(-t2))
p = [params[c] for c in cols]
params["fit_3m"], params["fit_2y"], params["fit_10y"] = nss(0.25, *p), nss(2.0, *p), nss(10.0, *p)
params["slope"] = params.fit_10y - params.fit_3m
params["curvature"] = 2 * params.fit_2y - params.fit_3m - params.fit_10y

In [ ]:
   #2 IMPORT
   import numpy as np, pandas as pd
   import plotly.graph_objects as go

In [ ]:
#3 A
import fredapi
if hasattr(fredapi.Fred, "_real_get"): fredapi.Fred.get_series = fredapi.Fred._real_get
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

SAMPLE_START = "1983-01-01"   # after the 1981-82 recession ended
WINDOW = 24                   # months: an inversion counts if a recession starts within this window

long = pd.DataFrame({s: fred.get_series(s, observation_start="1982-01-04")
                     for s in ["DGS10", "DGS2", "DGS3MO"]})
usrec = fred.get_series("USREC", observation_start="1982-01-01")
usrec.index = usrec.index + pd.offsets.MonthEnd(0)       # align monthly USREC to month-end

m = long.resample("ME").mean()                           # use "M" if your pandas rejects "ME"
m["s_10y2y"] = m.DGS10 - m.DGS2
m["s_10y3m"] = m.DGS10 - m.DGS3MO
m["rec"] = usrec
m = m.dropna(subset=["rec"])
m["rec_in_12m"] = m["rec"].shift(-12)
m = m.loc[SAMPLE_START:]

cur = long.dropna().iloc[-1]
cur_spread = {"s_10y2y": cur.DGS10 - cur.DGS2, "s_10y3m": cur.DGS10 - cur.DGS3MO}

rec_starts = m.index[m.rec.diff() == 1]

def months_between(a, b):
    return (b.to_period("M") - a.to_period("M")).n

def get_episodes(col, min_months=2):
    neg = m[col] < 0
    grp = (neg != neg.shift()).cumsum()
    eps = [(g.index[0], g.index[-1], len(g))
           for _, g in m[neg].groupby(grp[neg]) if len(g) >= min_months]
    return [e for e in eps if m.rec.loc[e[0]] == 0]      # skip inversions that begin during a recession

def inversion_table(col):
    last = m.index[-1]
    rows = []
    for s, e, n in get_episodes(col):
        nxt = rec_starts[rec_starts > s]
        if len(nxt):
            lead, nxt_txt = months_between(s, nxt[0]), nxt[0].strftime("%Y-%m")
        else:
            lead, nxt_txt = None, "none yet"
        if lead is not None and lead <= WINDOW:
            outcome = "Recession followed"
        elif months_between(s, last) < WINDOW:
            outcome = "Too recent to judge"
        else:
            outcome = f"No recession within {WINDOW}m"
        rows.append((s.strftime("%Y-%m"), e.strftime("%Y-%m"), n, nxt_txt, lead, outcome))
    df = pd.DataFrame(rows, columns=["Inversion start", "Inversion end", "Months inverted",
                                     "Next recession", "Lead (months)", "Outcome"])
    df["Lead (months)"] = df["Lead (months)"].astype("Int64")
    return df

for col in ["s_10y2y", "s_10y3m"]:
    d = m[[col, "rec_in_12m"]].dropna()
    res = sm.Probit(d["rec_in_12m"], sm.add_constant(d[col])).fit(
        disp=0, cov_type="HAC", cov_kwds={"maxlags": 12})
    auc = roc_auc_score(d["rec_in_12m"], -d[col])
    p_now = res.predict(np.array([[1.0, cur_spread[col]]]))[0]
    print(f"{col}: months={len(d)} | recession months={int(d.rec_in_12m.sum())} | "
          f"coef={res.params[col]:.2f} (p={res.pvalues[col]:.3f}) | pseudo R2={res.prsquared:.2f} | "
          f"AUC={auc:.2f} | latest spread={cur_spread[col]:.2f} -> implied prob {p_now:.0%}")
    print(inversion_table(col).to_string(index=False), "\n")

In [ ]:
# 3b
rec_m = m.copy(); print(type(rec_m))

In [ ]:
#4 B
import json
import plotly.io as pio
from sklearn.decomposition import PCA
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
FF = "Inter, Segoe UI, Helvetica, Arial, sans-serif"
C = ["#1d4ed8", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
tenors = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
CFG = {"displaylogo": False, "responsive": True}
PLO, PHI = 0, 100     # percentile range for the Parameters panels (0, 100 = full range; try 0.5, 99.5 to zoom in)

# ---------- helpers ----------
def base(fig, h, top=60):
    fig.update_layout(
        font=dict(family=FF, size=14, color="#1f2937"), height=h,
        margin=dict(l=85, r=175, t=top, b=85), plot_bgcolor="white",
        legend=dict(orientation="v", x=1.02, y=1, xanchor="left", yanchor="top",
                    bgcolor="rgba(255,255,255,0.95)", bordercolor="#e5e7eb",
                    borderwidth=1, font=dict(size=13)),
        hoverlabel=dict(font_size=13))
    ax = dict(showline=True, linecolor="#9ca3af", gridcolor="#eef0f3",
              tickfont=dict(size=13), title_font=dict(size=14), title_standoff=14)
    fig.update_xaxes(**ax)
    fig.update_yaxes(**ax)
    return fig

def left_titles(fig, n):
    names = ["xaxis"] + [f"xaxis{i}" for i in range(2, n + 1)]
    for a, nm in zip(fig.layout.annotations, names):
        a.update(x=fig.layout[nm].domain[0], xanchor="left", text=f"<b>{a.text}</b>",
                 font=dict(size=15, family=FF, color="#111827"))

def prng(*s, lo=PLO, hi=PHI, pad=0.1):
    v = np.concatenate([x.dropna().values for x in s])
    a, b = np.percentile(v, [lo, hi])
    p = (b - a) * pad or 0.1
    return [a - p, b + p]

def signed_loadings(model, names):
    L = pd.DataFrame(model.components_.T, index=names, columns=["PC1", "PC2", "PC3"])
    s = [np.sign(L["PC1"].sum()),
         np.sign(L.loc["DGS30", "PC2"] - L.iloc[0]["PC2"]),
         np.sign(L.loc["DGS2", "PC3"])]
    return L * s

# Legend tooltips: one listener on the whole chart, so every legend item is covered however it is redrawn
TIP_TEMPLATE = """
(function(){
  var gd = document.getElementById('{plot_id}');
  var TIPS = __TIPS__;
  var norm = function(s){ return (s || '').replace(/\\u00a0/g, ' ').replace(/\\s+/g, ' ').trim(); };
  var N = {}, seen = {};
  Object.keys(TIPS).forEach(function(k){ N[norm(k)] = TIPS[k]; });
  var tip = document.createElement('div');
  tip.style.cssText = 'position:fixed;display:none;max-width:330px;background:#0f172a;color:#fff;' +
    'padding:10px 12px;border-radius:8px;font:13px/1.5 Inter,Segoe UI,Arial,sans-serif;z-index:99999;' +
    'pointer-events:none;box-shadow:0 4px 14px rgba(0,0,0,.25)';
  document.body.appendChild(tip);
  function hide(){ tip.style.display = 'none'; }
  function place(ev){
    var x = ev.clientX + 16, y = ev.clientY + 16;
    if (x + 340 > window.innerWidth) x = ev.clientX - 346;
    tip.style.left = Math.max(8, x) + 'px';
    tip.style.top = y + 'px';
  }
  gd.addEventListener('mousemove', function(ev){
    var item = (ev.target && ev.target.closest) ? ev.target.closest('g.traces') : null;
    var t = item ? item.querySelector('.legendtext') : null;
    if (!t) { hide(); return; }
    var key = norm(t.textContent);
    if (N[key]) { tip.innerHTML = N[key]; tip.style.display = 'block'; place(ev); }
    else {
      hide();
      if (!seen[key]) { seen[key] = true; console.log('Legend tooltip: no text for legend item', JSON.stringify(key)); }
    }
  }, true);
  gd.addEventListener('mouseleave', hide);
})();
"""
tip_js = lambda d: TIP_TEMPLATE.replace("__TIPS__", json.dumps(d))

# ---------- 1. NSS fits ----------
last = str(yields.index[-1].date())
f1 = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                   subplot_titles=("2007-03-01 (inverted curve)", f"{last} (latest)"))
g = np.linspace(0.08, 30, 300)
for k, dt in enumerate(["2007-03-01", last], start=1):
    f1.add_scatter(x=mats, y=yields.loc[dt].values, mode="markers", name="Observed yield",
                   marker=dict(size=10, color=C[1]), showlegend=(k == 1), row=1, col=k)
    f1.add_scatter(x=g, y=nss(g, *params.loc[dt, cols].values.astype(float)), mode="lines",
                   name="NSS fit", line=dict(color=C[0], width=3), showlegend=(k == 1), row=1, col=k)
    f1.update_xaxes(type="log", tickvals=mats, ticktext=tenors, title_text="<b>Maturity</b>", row=1, col=k)
    f1.update_yaxes(title_text="<b>Yield (%)</b>", row=1, col=k)
base(f1, 470); left_titles(f1, 2)

# ---------- 2. Parameters and fit quality ----------
f2 = make_subplots(rows=4, cols=1, shared_xaxes=False, vertical_spacing=0.085,
                   subplot_titles=("NSS betas", "NSS lambdas (decay parameters)",
                                   "Model-implied slope (10y-3m) and curvature", "Daily fit error (RMSE)"))
leg = ["legend", "legend2", "legend3", "legend4"]
for i, c in enumerate(["b0", "b1", "b2", "b3"]):
    f2.add_scatter(x=params.index, y=params[c], name=c, legend=leg[0],
                   line=dict(color=C[i], width=1.6), row=1, col=1)
for i, c in enumerate(["l1", "l2"]):
    f2.add_scatter(x=params.index, y=params[c], name=c, legend=leg[1],
                   line=dict(color=C[i + 4], width=1.6), row=2, col=1)
for i, (c, lbl) in enumerate([("slope", "slope (10y-3m)"), ("curvature", "curvature")]):
    f2.add_scatter(x=params.index, y=params[c], name=lbl, legend=leg[2],
                   line=dict(color=C[i], width=1.6), row=3, col=1)
f2.add_hline(y=0, line_dash="dash", line_color="#6b7280", row=3, col=1)
f2.add_scatter(x=params.index, y=params.rmse * 100, name="RMSE (bp)", legend=leg[3],
               line=dict(color="#374151", width=1.2), row=4, col=1)
f2.add_hline(y=5, line_dash="dash", line_color=C[1], row=4, col=1,
             annotation_text="5 bp threshold", annotation_position="top left",
             annotation_font=dict(size=12, color=C[1]))
f2.update_yaxes(range=prng(params.b0, params.b1, params.b2, params.b3), title_text="<b>Percent</b>", row=1, col=1)
f2.update_yaxes(range=prng(params.l1, params.l2), title_text="<b>Years</b>", row=2, col=1)
f2.update_yaxes(range=prng(params.slope, params.curvature), title_text="<b>Pct. points</b>", row=3, col=1)
f2.update_yaxes(range=[0, params.rmse.max() * 100 * 1.12], title_text="<b>bp</b>", row=4, col=1)
f2.update_xaxes(range=[params.index.min(), params.index.max()], showticklabels=True, matches="x",
                title_text="<b>Date</b>")
base(f2, 1250); left_titles(f2, 4)
for k, name in enumerate(leg):
    top = f2.layout[f"yaxis{'' if k == 0 else k + 1}"].domain[1]
    f2.layout[name] = dict(x=1.02, y=top, xanchor="left", yanchor="top",
                           bgcolor="rgba(255,255,255,0.95)", bordercolor="#e5e7eb",
                           borderwidth=1, font=dict(size=13))

# tooltip text for section 2 (numbers come from your fitted data)
bound = {c: int(((params[c] <= lower[i] + 1e-3) | (params[c] >= upper[i] - 1e-3)).sum())
         for i, c in enumerate(cols)}
stat = lambda c: (f"<br>Mean {params[c].mean():.2f}, range {params[c].min():.2f} to {params[c].max():.2f}. "
                  f"At an optimiser bound on {bound[c]:,} of {len(params):,} days.")
neg_share = (params.slope < 0).mean() * 100
tips2 = {
    "b0": "<b>β0 · long-run level</b><br>Where the curve settles at very long maturities. Weakly identified when "
          "β2 and β3 offset it, so read it with the fitted yields rather than as the 30-year rate." + stat("b0"),
    "b1": "<b>β1 · slope</b><br>The short-end yield is roughly β0 + β1, so a negative β1 means an "
          "upward-sloping curve and a positive β1 an inverted one." + stat("b1"),
    "b2": "<b>β2 · first hump</b><br>Size and direction of a hump or dip in the medium maturities, "
          "positioned by λ1." + stat("b2"),
    "b3": "<b>β3 · second hump</b><br>The extra hump that makes this the Svensson extension, positioned by λ2. "
          "A small penalty in the fit stops β2 and β3 from offsetting each other." + stat("b3"),
    "l1": f"<b>λ1 · first decay</b><br>Sets where β2's hump peaks, at roughly 1.8 × λ1 years. "
          f"Allowed range {lower[4]:g} to {upper[4]:g}." + stat("l1"),
    "l2": f"<b>λ2 · second decay</b><br>Sets where β3's hump peaks, at roughly 1.8 × λ2 years. "
          f"Allowed range {lower[5]:g} to {upper[5]:g}." + stat("l2"),
    "slope (10y-3m)": f"<b>Slope (10y − 3m)</b><br>Fitted 10-year yield minus fitted 3-month yield, in percentage "
                      f"points. Negative means the curve is inverted, which it was on {neg_share:.0f}% of days.",
    "curvature": "<b>Curvature</b><br>2 × fitted 2y − fitted 3m − fitted 10y. Positive when the middle of the "
                 "curve sits above the line joining the short and long ends.",
    "RMSE (bp)": f"<b>Fit error (RMSE)</b><br>Root-mean-square gap between fitted and observed yields across "
                 f"the 11 maturities that day. Mean {params.rmse.mean()*100:.1f} bp, max "
                 f"{params.rmse.max()*100:.1f} bp, {(params.rmse > 0.05).sum():,} days above 5 bp.",
}

# ---------- 3. PCA ----------
dy = yields.diff().dropna() * 100
cols2 = [c for c in yields.columns if c not in ("DGS1MO", "DGS3MO")]
pcaA = PCA(n_components=3).fit(dy)
pcaB = PCA(n_components=3).fit(dy[cols2])
LA, LB = signed_loadings(pcaA, list(yields.columns)), signed_loadings(pcaB, cols2)
f3 = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.36, 0.28], horizontal_spacing=0.09,
                   subplot_titles=("Loadings: all 11 maturities", "Loadings: excluding 1M and 3M",
                                   "Variance explained"))
for i, c in enumerate(["PC1", "PC2", "PC3"]):
    f3.add_scatter(x=tenors, y=LA[c], mode="lines+markers", name=c, legendgroup="load",
                   legendgrouptitle_text="Loadings", line=dict(color=C[i], width=3), row=1, col=1)
    f3.add_scatter(x=tenors[2:], y=LB[c], mode="lines+markers", name=c, legendgroup="load",
                   showlegend=False, line=dict(color=C[i], width=3), row=1, col=2)
f3.add_bar(x=["PC1", "PC2", "PC3"], y=pcaA.explained_variance_ratio_ * 100, name="All 11 maturities",
           legendgroup="var", legendgrouptitle_text="Variance explained (%)", marker_color="#1e293b", row=1, col=3)
f3.add_bar(x=["PC1", "PC2", "PC3"], y=pcaB.explained_variance_ratio_ * 100, name="Excl. 1M and 3M",
           legendgroup="var", marker_color="#94a3b8", row=1, col=3)
f3.update_xaxes(title_text="<b>Maturity</b>", row=1, col=1)
f3.update_xaxes(title_text="<b>Maturity</b>", row=1, col=2)
f3.update_xaxes(title_text="<b>Component</b>", row=1, col=3)
f3.update_yaxes(title_text="<b>Loading</b>", row=1, col=1)
f3.update_yaxes(title_text="<b>% of variance</b>", row=1, col=3)
f3.update_layout(barmode="group")
base(f3, 500); left_titles(f3, 3)

vA, vB = pcaA.explained_variance_ratio_ * 100, pcaB.explained_variance_ratio_ * 100
tips3 = {
    "PC1": f"<b>PC1 · level</b><br>Yields at all maturities move in the same direction. Explains {vA[0]:.0f}% of "
           f"daily yield-change variance with all 11 maturities and {vB[0]:.0f}% excluding 1M and 3M.",
    "PC2": f"<b>PC2 · slope</b><br>Short and long maturities move in opposite directions, so the curve steepens "
           f"or flattens. Explains {vA[1]:.0f}% (all 11) and {vB[1]:.0f}% (excluding 1M and 3M).",
    "PC3": f"<b>PC3 · curvature</b><br>The middle of the curve moves against the short and long ends. "
           f"Explains {vA[2]:.0f}% (all 11) and {vB[2]:.0f}% (excluding 1M and 3M).",
    "All 11 maturities": f"<b>All 11 maturities</b><br>PCA on daily yield changes in basis points. The three "
                         f"components together explain {vA.sum():.1f}% of variance. The 1M and 3M bills are "
                         f"volatile and carry the largest loadings on PC2 and PC3.",
    "Excl. 1M and 3M": f"<b>Excluding 1M and 3M</b><br>The same PCA without the two shortest bills. The three "
                       f"components explain {vB.sum():.1f}% of variance, with a cleaner level / slope / "
                       f"curvature split.",
}

# ---------- 4. Spreads vs recessions ----------
f4 = go.Figure()
f4.add_scatter(x=m.index, y=m.s_10y2y, name="10y-2y spread", line=dict(color=C[0], width=2))
f4.add_scatter(x=m.index, y=m.s_10y3m, name="10y-3m spread", line=dict(color=C[1], width=2))
f4.add_scatter(x=[None], y=[None], mode="markers", name="NBER recession",
               marker=dict(size=13, color="#9ca3af", symbol="square"))
f4.add_hline(y=0, line_dash="dash", line_color="#111827")
is_rec = m.rec == 1
gr = (m.rec != m.rec.shift()).cumsum()
for _, grp in m[is_rec].groupby(gr[is_rec]):
    f4.add_vrect(x0=grp.index[0], x1=grp.index[-1], fillcolor="#9ca3af", opacity=0.35, line_width=0)
f4.update_xaxes(title_text="<b>Date</b>")
f4.update_yaxes(title_text="<b>Spread (percentage points)</b>")
base(f4, 500, top=30)

# ---------- 5. Surface: fixed bold labels, turntable, zoom cap, pan lock ----------
mp = params.resample("ME").last()[cols].dropna()
yrs = (mp.index.year + (mp.index.month - 1) / 12).values
grid = np.geomspace(mats[0], mats[-1], 120)
xpos = np.interp(grid, mats, np.arange(len(mats)))
Z = np.array([nss(grid, *row) for row in mp.values.astype(float)])
nx, ymin, ymax = len(mats) - 1, yrs.min(), yrs.max()
year_ticks = list(range(2002, 2027, 4))

def lab(x, y, z, t, size=13):
    return dict(x=x, y=y, z=z, text=f"<b>{t}</b>", showarrow=False,
                font=dict(size=size, color="#111827", family=FF))

ann = [lab(i, ymin - 1.8, 0, t) for i, t in enumerate(tenors)]
ann += [lab(nx + 1.0, yv, 0, yv) for yv in year_ticks]
ann += [lab(-1.0, ymin - 1.8, v, f"{v}%") for v in range(0, 7)]
ann += [lab(nx / 2, ymin - 4.6, 0, "Maturity", 16),
        lab(nx + 2.9, (ymin + ymax) / 2, 0, "Year", 16),
        lab(-2.3, ymin - 1.8, 3.25, "Yield (%)", 16)]

cam = dict(eye=dict(x=-1.8, y=-1.8, z=0.9), center=dict(x=0, y=0, z=0), up=dict(x=0, y=0, z=1))
hide = dict(showticklabels=False, ticks="", title_text="", showspikes=False)
f5 = go.Figure(go.Surface(x=xpos, y=yrs, z=Z, colorscale="Viridis", cmin=0, cmax=6.5,
                          colorbar=dict(title="Yield (%)", len=0.7),
                          hovertemplate="Year %{y:.1f}<br>Yield %{z:.2f}%<extra></extra>"))
f5.update_layout(
    font=dict(family=FF, size=14), height=760, margin=dict(l=0, r=0, t=20, b=0), uirevision="keep",
    updatemenus=[dict(type="buttons", x=0.02, y=0.98, buttons=[dict(
        label="Reset view", method="relayout", args=[{"scene.camera": cam}])])],
    scene=dict(
        dragmode="turntable", camera=cam, annotations=ann,
        xaxis=dict(tickvals=list(range(len(mats))), range=[-2.8, nx + 3.6], **hide),
        yaxis=dict(tickvals=year_ticks, range=[ymin - 5.2, ymax + 0.5], **hide),
        zaxis=dict(tickvals=list(range(7)), range=[0, 6.5], **hide),
        aspectmode="manual", aspectratio=dict(x=1.5, y=1.9, z=0.8)))

ZOOM_JS = """
(function(){
  var gd = document.getElementById('{plot_id}');
  var MAXD = 3.2, MIND = 1.5, busy = false;
  function clamp(){
    if (busy) return;
    var sc = gd._fullLayout && gd._fullLayout.scene, cam = null;
    try { cam = sc._scene.getCamera(); } catch(e) {}
    if (!cam && sc) cam = sc.camera;
    if (!cam || !cam.eye) return;
    var e = cam.eye, c = cam.center || {x:0, y:0, z:0};
    var d = Math.sqrt(e.x*e.x + e.y*e.y + e.z*e.z);
    var off = Math.abs(c.x) > 0.01 || Math.abs(c.y) > 0.01 || Math.abs(c.z) > 0.01;
    if (d > MAXD || d < MIND || off) {
      var k = d > MAXD ? MAXD / d : (d < MIND ? MIND / d : 1);
      busy = true;
      Plotly.relayout(gd, {'scene.camera.eye': {x: e.x*k, y: e.y*k, z: e.z*k},
                           'scene.camera.center': {x: 0, y: 0, z: 0}})
            .then(function(){ busy = false; });
    }
  }
  gd.on('plotly_relayout', clamp);
  gd.on('plotly_relayouting', clamp);
})();
"""

# ---------- 6. Heatmap ----------
f6 = go.Figure(go.Heatmap(x=xpos, y=mp.index, z=Z, colorscale="Viridis", zmin=0, zmax=6.5,
                          colorbar=dict(title="Yield (%)"),
                          hovertemplate="%{y|%b %Y}<br>Yield %{z:.2f}%<extra></extra>"))
base(f6, 700, top=30)
f6.update_layout(
    xaxis=dict(tickvals=list(range(len(mats))), ticktext=tenors, title_text="<b>Maturity</b>"),
    yaxis=dict(title_text="<b>Year</b>", dtick="M48", tickformat="%Y"))

# ---------- tables ----------
pfmt = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
rows = []
for col, name in [("s_10y2y", "10y-2y"), ("s_10y3m", "10y-3m")]:
    d = m[[col, "rec_in_12m"]].dropna()
    r = sm.Probit(d["rec_in_12m"], sm.add_constant(d[col])).fit(
        disp=0, cov_type="HAC", cov_kwds={"maxlags": 12})
    rows.append({"Spread": name, "Months": len(d), "Coefficient": round(r.params[col], 2),
                 "p-value (HAC)": pfmt(r.pvalues[col]), "Pseudo R²": round(r.prsquared, 2),
                 "AUC": round(roc_auc_score(d["rec_in_12m"], -d[col]), 2),
                 "Latest spread (pp)": round(cur_spread[col], 2),
                 "Implied 12m recession probability":
                     f"{r.predict(np.array([[1.0, cur_spread[col]]]))[0]:.0%}"})

tbl = lambda df: df.to_html(index=False, border=0, classes="tbl", na_rep="n/a")
t_probit = tbl(pd.DataFrame(rows))
ep_block = lambda title, col: f"<h3>{title}</h3>{tbl(inversion_table(col))}"

# ---------- page ----------
cards = [("Days fitted", f"{len(params):,}"), ("Mean fit error", f"{params.rmse.mean()*100:.1f} bp"),
         ("Days over 5 bp", f"{(params.rmse > 0.05).sum()}"),
         ("PC1 variance (excl. 1M/3M)", f"{vB[0]:.0f}%"),
         ("10y-3m recession AUC", f"{rows[1]['AUC']:.2f}"),
         ("Latest 10y-3m spread", f"{cur_spread['s_10y3m']:.2f} pp")]
cards_html = "".join(f'<div class="card"><div class="v">{v}</div><div class="k">{k}</div></div>'
                     for k, v in cards)

def section(n, sid, title, text, fig, first=False, post=None):
    div = fig.to_html(full_html=False, include_plotlyjs=True if first else False,
                      config=CFG, post_script=post)
    return (f'<section id="{sid}"><div class="sh"><span class="num">{n:02d}</span><div>'
            f'<h2>{title}</h2><p class="sub">{text}</p></div></div>{div}</section>')

body = (
    section(1, "fit", "Curve Fitting",
            "Six-parameter Nelson-Siegel-Svensson curves fitted by least squares to the 11 constant-maturity "
            "Treasury yields each day, warm-started from the previous day's solution.", f1, first=True)
  + section(2, "params", "Parameters and Fit Quality",
            "Betas and lambdas trade off against each other, so level, slope and curvature are taken from "
            "model-implied yields. A parameter sits at its optimiser bound on about a quarter of days with no "
            "loss of fit, so the analysis relies on fitted yields rather than raw parameters. "
            "Fit errors rise in 2001-02, 2008-09 and 2022-23. Hover over a legend item for an explanation.",
            f2, post=tip_js(tips2))
  + section(3, "pca", "Principal Components",
            "Covariance PCA on daily yield changes. Excluding the 1M and 3M bills removes the noisiest part "
            "of the curve and gives a cleaner level / slope / curvature split. Component signs are "
            "normalised to a common convention. Hover over a legend item for an explanation.",
            f3, post=tip_js(tips3))
  + section(4, "recession", "Curve Inversion and Recessions",
            "Spreads against NBER recession periods (grey). Predictive power is measured with a probit of "
            "a recession 12 months ahead, using HAC standard errors for overlapping observations. "
            "The sample starts in January 1983, after the 1981-82 recession ended.", f4)
  + f'<section id="tests"><div class="sh"><span class="num">05</span><div><h2>Recession Test Results</h2>'
    f'<p class="sub">Probit estimates, then every inversion episode, the next recession and the lead time.</p></div></div>'
    f'<h3>Predictive power</h3>{t_probit}'
    f'{ep_block("Inversion episodes: 10y-2y", "s_10y2y")}'
    f'{ep_block("Inversion episodes: 10y-3m", "s_10y3m")}'
    f'<p class="note">The sample contains four recessions, so these results are suggestive rather than '
    f'conclusive. Inversions that begin during a recession are excluded. "Outcome" counts a recession as '
    f'following if one starts within {WINDOW} months of the inversion beginning, so several episodes can '
    f'point to the same recession. The 2020 recession was triggered by COVID.</p></section>'
  + section(6, "surface", "Yield Surface",
            "Fitted yield by maturity and year. Drag to rotate around the centre, scroll to zoom (zoom range is "
            "limited and panning is disabled). Labels are fixed and the vertical axis cannot flip.", f5, post=ZOOM_JS)
  + section(7, "heat", "Yield Curve Heatmap",
            "The same fitted surface as a flat map. Short maturities are on the left; an inversion shows "
            "as a brighter left side than right.", f6))

css = """*{box-sizing:border-box}body{margin:0;font-family:Inter,'Segoe UI',Helvetica,Arial,sans-serif;background:#f3f4f6;color:#1f2937}
header{background:linear-gradient(135deg,#0f172a,#1e3a8a);color:#fff;padding:44px 8vw 30px}
header h1{margin:0 0 8px;font-size:34px;letter-spacing:-.5px}header .tag{margin:0 0 20px;color:#cbd5e1;font-size:16px}
nav a{color:#e0e7ff;text-decoration:none;font-size:14px;margin-right:18px;border-bottom:1px solid rgba(224,231,255,.4);padding-bottom:2px}
nav a:hover{color:#fff;border-color:#fff}
main{max-width:1240px;margin:0 auto;padding:12px 24px 40px}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;margin:24px 0}
.card{background:#fff;border-radius:12px;padding:18px;box-shadow:0 1px 3px rgba(0,0,0,.1)}
.v{font-size:28px;font-weight:700;color:#1d4ed8}.k{color:#6b7280;margin-top:4px;font-size:14px}
section{background:#fff;border-radius:12px;padding:26px 30px;margin:24px 0;box-shadow:0 1px 3px rgba(0,0,0,.1);overflow:hidden}
.sh{display:flex;gap:16px;align-items:flex-start;margin-bottom:14px;padding-bottom:14px;border-bottom:1px solid #e5e7eb}
.num{font-size:15px;font-weight:700;color:#fff;background:#1d4ed8;border-radius:8px;padding:6px 10px;min-width:42px;text-align:center}
h2{margin:0;font-size:24px;color:#0f172a}.sub{margin:4px 0 0;font-size:15px;line-height:1.55;color:#4b5563;max-width:900px}
h3{margin:26px 0 8px;font-size:17px;color:#0f172a}
.tbl{border-collapse:collapse;width:100%;font-size:15px}.tbl th{background:#f3f4f6;text-align:left;padding:10px 12px;color:#374151}
.tbl td{padding:9px 12px;border-top:1px solid #e5e7eb}
.note{font-size:14px;color:#6b7280;line-height:1.6;margin-top:16px}
footer{max-width:1240px;margin:0 auto;padding:8px 24px 40px;color:#6b7280;font-size:13px;line-height:1.6}"""

nav = ('<nav><a href="#fit">Curve fitting</a><a href="#params">Parameters</a><a href="#pca">PCA</a>'
       '<a href="#recession">Recessions</a><a href="#tests">Results</a><a href="#surface">Surface</a>'
       '<a href="#heat">Heatmap</a></nav>')
footer = ("<footer>Data: Federal Reserve Bank of St. Louis (FRED), constant-maturity Treasury yields DGS1MO to DGS30 "
          "and the NBER recession indicator USREC. Method: daily Nelson-Siegel-Svensson least-squares fits, "
          "covariance PCA on daily changes, probit recession tests (HAC standard errors). "
          "Educational project; not investment advice.</footer>")

html = ("<!doctype html><html lang='en'><head><meta charset='utf-8'>"
        "<meta name='viewport' content='width=device-width,initial-scale=1'><title>Yield Curve Lab</title>"
        f"<style>{css}</style></head><body><header><h1>Yield Curve Lab</h1>"
        "<p class='tag'>Nelson-Siegel-Svensson fitting, principal components and recession signals "
        "for US Treasuries, July 2001 to September 2026</p>"
        f"{nav}</header><main><div class='cards'>{cards_html}</div>{body}</main>{footer}</body></html>")

with open("yield_curve_lab.html", "w", encoding="utf-8") as fh:
    fh.write(html)
print("Saved, size: %.1f MB" % (len(html) / 1e6))

from google.colab import files
files.download("yield_curve_lab.html")

In [ ]:
#5 C
import json

# ---------- hide Plotly's own legends in sections 2 and 3 ----------
f2.update_layout(showlegend=False)
f3.update_layout(showlegend=False)

# ---------- HTML legend builders ----------
def item(plot_id, label, color, traces, tip, square=False):
    cls = "sw sq" if square else "sw"
    tr = json.dumps(traces)
    return ('<div class="lg" data-plot="' + plot_id + '" data-traces=\'' + tr + '\'>'
            '<span class="' + cls + '" style="background:' + color + '"></span>'
            '<span class="lt">' + label + '</span><div class="tt">' + tip + '</div></div>')

gt = lambda t, first=True: f'<div class="lgt{"" if first else " lgt2"}">{t}</div>'
box = lambda top, parts: f'<div class="lgbox" style="top:{top:.0f}px">' + "".join(parts) + '</div>'

# Section 2: one legend box beside each panel (positions come from the figure layout)
H2, MT2, MB2 = f2.layout.height, f2.layout.margin.t, f2.layout.margin.b
dom_top = [f2.layout[f"yaxis{'' if k == 0 else k + 1}"].domain[1] for k in range(4)]
top2 = [MT2 + (1 - d) * (H2 - MT2 - MB2) + 2 for d in dom_top]
P2 = "plot_params"
leg2 = (
    box(top2[0], [gt("Betas")] + [item(P2, c, C[i], [i], tips2[c])
                                  for i, c in enumerate(["b0", "b1", "b2", "b3"])])
  + box(top2[1], [gt("Lambdas")] + [item(P2, c, C[i + 4], [4 + i], tips2[c])
                                    for i, c in enumerate(["l1", "l2"])])
  + box(top2[2], [gt("Slope and curvature"),
                  item(P2, "slope (10y-3m)", C[0], [6], tips2["slope (10y-3m)"]),
                  item(P2, "curvature", C[1], [7], tips2["curvature"])])
  + box(top2[3], [gt("Fit error"), item(P2, "RMSE (bp)", "#374151", [8], tips2["RMSE (bp)"])])
)

# Section 3: one legend box beside the three panels
P3 = "plot_pca"
top3 = f3.layout.margin.t + 2
leg3 = box(top3,
           [gt("Loadings")]
           + [item(P3, f"PC{i + 1}", C[i], [2 * i, 2 * i + 1], tips3[f"PC{i + 1}"]) for i in range(3)]
           + [gt("Variance explained (%)", first=False),
              item(P3, "All 11 maturities", "#1e293b", [6], tips3["All 11 maturities"], square=True),
              item(P3, "Excl. 1M and 3M", "#94a3b8", [7], tips3["Excl. 1M and 3M"], square=True)])

# ---------- page ----------
def section(n, sid, title, text, fig, first=False, post=None, legend_html="", div_id=None):
    div = fig.to_html(full_html=False, include_plotlyjs=True if first else False,
                      config=CFG, post_script=post, div_id=div_id)
    return (f'<section id="{sid}"><div class="sh"><span class="num">{n:02d}</span><div>'
            f'<h2>{title}</h2><p class="sub">{text}</p></div></div>'
            f'<div class="plotwrap">{div}{legend_html}</div></section>')

body = (
    section(1, "fit", "Curve Fitting",
            "Six-parameter Nelson-Siegel-Svensson curves fitted by least squares to the 11 constant-maturity "
            "Treasury yields each day, warm-started from the previous day's solution.", f1, first=True)
  + section(2, "params", "Parameters and Fit Quality",
            "Betas and lambdas trade off against each other, so level, slope and curvature are taken from "
            "model-implied yields. A parameter sits at its optimiser bound on about a quarter of days with no "
            "loss of fit, so the analysis relies on fitted yields rather than raw parameters. "
            "Fit errors rise in 2001-02, 2008-09 and 2022-23. Hover over a legend item for an explanation; "
            "click an item to show or hide its line.",
            f2, legend_html=leg2, div_id=P2)
  + section(3, "pca", "Principal Components",
            "Covariance PCA on daily yield changes. Excluding the 1M and 3M bills removes the noisiest part "
            "of the curve and gives a cleaner level / slope / curvature split. Component signs are "
            "normalised to a common convention. Hover over a legend item for an explanation; "
            "click an item to show or hide it.",
            f3, legend_html=leg3, div_id=P3)
  + section(4, "recession", "Curve Inversion and Recessions",
            "Spreads against NBER recession periods (grey). Predictive power is measured with a probit of "
            "a recession 12 months ahead, using HAC standard errors for overlapping observations. "
            "The sample starts in January 1983, after the 1981-82 recession ended.", f4)
  + f'<section id="tests"><div class="sh"><span class="num">05</span><div><h2>Recession Test Results</h2>'
    f'<p class="sub">Probit estimates, then every inversion episode, the next recession and the lead time.</p></div></div>'
    f'<h3>Predictive power</h3>{t_probit}'
    f'{ep_block("Inversion episodes: 10y-2y", "s_10y2y")}'
    f'{ep_block("Inversion episodes: 10y-3m", "s_10y3m")}'
    f'<p class="note">The sample contains four recessions, so these results are suggestive rather than '
    f'conclusive. Inversions that begin during a recession are excluded. "Outcome" counts a recession as '
    f'following if one starts within {WINDOW} months of the inversion beginning, so several episodes can '
    f'point to the same recession. The 2020 recession was triggered by COVID.</p></section>'
  + section(6, "surface", "Yield Surface",
            "Fitted yield by maturity and year. Drag to rotate around the centre, scroll to zoom (zoom range is "
            "limited and panning is disabled). Labels are fixed and the vertical axis cannot flip.",
            f5, post=ZOOM_JS)
  + section(7, "heat", "Yield Curve Heatmap",
            "The same fitted surface as a flat map. Short maturities are on the left; an inversion shows "
            "as a brighter left side than right.", f6))

css2 = css + """
.plotwrap{position:relative}
.lgbox{position:absolute;right:8px;width:158px;background:rgba(255,255,255,.96);border:1px solid #e5e7eb;
  border-radius:6px;padding:8px 10px;z-index:5}
.lgt{font-size:12px;font-weight:700;color:#111827;margin-bottom:4px}
.lgt2{margin-top:10px}
.lg{position:relative;display:flex;align-items:center;gap:8px;padding:3px 2px;cursor:pointer;font-size:13px;color:#1f2937}
.lg.off{opacity:.35}
.sw{width:22px;height:3px;display:inline-block;border-radius:2px;flex:none}
.sw.sq{width:12px;height:12px;margin:0 5px}
.tt{display:none;position:absolute;right:calc(100% + 14px);top:-6px;width:310px;background:#0f172a;color:#fff;
  padding:10px 12px;border-radius:8px;font-size:13px;line-height:1.5;z-index:99;cursor:default;
  box-shadow:0 4px 14px rgba(0,0,0,.25)}
.lg:hover .tt{display:block}
"""

toggle_js = """<script>
document.querySelectorAll('.lg[data-plot]').forEach(function(el){
  el.addEventListener('click', function(){
    var gd = document.getElementById(el.getAttribute('data-plot'));
    var idx = JSON.parse(el.getAttribute('data-traces'));
    var off = el.classList.toggle('off');
    Plotly.restyle(gd, {visible: !off}, idx);
  });
});
</script>"""

html = ("<!doctype html><html lang='en'><head><meta charset='utf-8'>"
        "<meta name='viewport' content='width=device-width,initial-scale=1'><title>Yield Curve Lab</title>"
        f"<style>{css2}</style></head><body><header><h1>Yield Curve Lab</h1>"
        "<p class='tag'>Nelson-Siegel-Svensson fitting, principal components and recession signals "
        "for US Treasuries, July 2001 to September 2026</p>"
        f"{nav}</header><main><div class='cards'>{cards_html}</div>{body}</main>{footer}{toggle_js}</body></html>")

with open("yield_curve_lab.html", "w", encoding="utf-8") as fh:
    fh.write(html)
print("Saved, size: %.1f MB" % (len(html) / 1e6))

from google.colab import files
files.download("yield_curve_lab.html")

In [ ]:
# 6 PATCH
import fredapi
if not hasattr(fredapi.Fred, "_real_get"):
    fredapi.Fred._real_get = fredapi.Fred.get_series            # saved once, so rerunning is safe
_orig = fredapi.Fred._real_get
try: G
except NameError: G = pd.read_csv("gsw_zero.csv", index_col=0, parse_dates=True)
def _gs(self, sid, *a, **k):
    if sid.startswith("DGS") and sid[3:].isdigit():             # Treasury par request -> Fed zero-coupon yield
        return G[f"SVENY{int(sid[3:]):02d}"].loc[k.get("observation_start", "2001-07-31"):].dropna()
    return _orig(self, sid, *a, **k)
fredapi.Fred.get_series = _gs

In [ ]:
#7 I
!pip install fredapi -q
import io, os, requests, numpy as np, pandas as pd, statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from fredapi import Fred
from google.colab import userdata, files

# ---------- data ----------
fred = Fred(api_key=userdata.get("FRED_API_KEY"))
tk = [1, 2, 3, 5, 7, 10, 20]
spot = pd.read_csv("gilt_spot_curve.csv", index_col=0, parse_dates=True); spot.columns = spot.columns.astype(float)
us = pd.DataFrame({f"{t}Y": fred.get_series(f"DGS{t}", observation_start="2001-07-31") for t in tk}).dropna()
uk = spot[[float(t) for t in tk]].loc["2001-07-31":].dropna(); uk.columns = us.columns
idx = us.index.intersection(uk.index); us, uk = us.loc[idx], uk.loc[idx]
wu, wk = (d.resample("W-FRI").last().diff().dropna() * 100 for d in (us, uk))      # weekly changes, bp
j = pd.concat([wk["10Y"].rename("gilt"), wu["10Y"].rename("ust")], axis=1)

FF = "Inter, Segoe UI, Helvetica, Arial, sans-serif"; C = ["#1d4ed8", "#dc2626"]
CFG = {"displaylogo": False, "responsive": True}
def sty(f, h, top=60):
    f.update_layout(font=dict(family=FF, size=14, color="#1f2937"), height=h, plot_bgcolor="white",
                    margin=dict(l=85, r=150, t=top, b=80),
                    legend=dict(x=1.02, y=1, xanchor="left", bgcolor="rgba(255,255,255,.95)",
                                bordercolor="#e5e7eb", borderwidth=1))
    ax = dict(showline=True, linecolor="#9ca3af", gridcolor="#eef0f3", title_standoff=14)
    f.update_xaxes(**ax); f.update_yaxes(**ax); return f

# ---------- figures ----------
rc = j.gilt.rolling(52).corr(j.ust).dropna(); full = j.gilt.corr(j.ust)
f1 = go.Figure(go.Scatter(x=rc.index, y=rc, line=dict(color=C[0], width=2)))
f1.add_hline(y=full, line_dash="dash", line_color="#6b7280", annotation_text=f"full sample {full:.2f}",
             annotation_position="bottom right")
f1.update_xaxes(title_text="<b>Date</b>"); f1.update_yaxes(title_text="<b>Correlation</b>", range=[0.3, 1])
sty(f1, 420, 30).update_layout(showlegend=False)

gap = uk["10Y"] - us["10Y"]
f2 = go.Figure(go.Scatter(x=gap.index, y=gap, line=dict(color=C[1], width=1.5)))
f2.add_hline(y=0, line_dash="dash", line_color="#111827")
for d, n in [("2009-03-05", "BoE QE begins"), ("2016-06-23", "Brexit vote"), ("2022-09-23", "Mini-budget")]:
    f2.add_shape(type="line", x0=d, x1=d, y0=0, y1=1, yref="paper", line=dict(color="#6b7280", dash="dot"))
    f2.add_annotation(x=d, y=1, yref="paper", text=f"<b>{n}</b>", showarrow=False, yanchor="bottom")
f2.update_xaxes(title_text="<b>Date</b>"); f2.update_yaxes(title_text="<b>Gilt minus Treasury, 10y (pp)</b>")
sty(f2, 450).update_layout(showlegend=False)

def pc(d):
    dy = d.diff().dropna() * 100
    p = PCA(3).fit(dy)
    L = pd.DataFrame(p.components_.T, index=d.columns, columns=["PC1", "PC2", "PC3"])
    s = np.array([np.sign(L.PC1.sum()), np.sign(L.loc["20Y", "PC2"] - L.loc["1Y", "PC2"]), np.sign(L.loc["5Y", "PC3"])])
    return p.explained_variance_ratio_ * 100, L * s, pd.DataFrame(p.transform(dy) * s, index=dy.index, columns=L.columns)
vu, Lu, Su = pc(us); vk, Lk, Sk = pc(uk)
wsc = lambda S: S.resample("W-FRI").sum()
cs = [wsc(Su)[c].corr(wsc(Sk)[c]) for c in Su.columns]
f3 = make_subplots(rows=1, cols=3, horizontal_spacing=0.07,
                   subplot_titles=["PC1: level", "PC2: slope", "PC3: curvature"])
for i, c in enumerate(Lu.columns, 1):
    for L, n, col in ((Lu, "US", C[0]), (Lk, "UK", C[1])):
        f3.add_scatter(x=L.index, y=L[c], mode="lines+markers", name=n, legendgroup=n, showlegend=(i == 1),
                       line=dict(color=col, width=3), row=1, col=i)
    f3.update_xaxes(title_text="<b>Maturity</b>", row=1, col=i)
f3.update_yaxes(title_text="<b>Loading</b>", row=1, col=1)
sty(f3, 450)
for a, k in zip(f3.layout.annotations, ["xaxis", "xaxis2", "xaxis3"]):
    a.update(x=f3.layout[k].domain[0], xanchor="left", text=f"<b>{a.text}</b>", font=dict(size=15))

# ---------- tables ----------
per = {"2001-07": ("2001", "2007"), "2008-15": ("2008", "2015"), "2016-21": ("2016", "2021"), "2022-now": ("2022", "2026")}
tn = ["1Y", "2Y", "5Y", "10Y", "20Y"]
t_corr = pd.DataFrame({k: [wu.loc[a:b, t].corr(wk.loc[a:b, t]) for t in tn] for k, (a, b) in per.items()},
                      index=tn).T.round(2).rename_axis("Period").reset_index()

w = j.copy(); w["post"] = (w.index >= "2022-01-01").astype(float); w["ust_post"] = w.ust * w.post
def bt(x, label):
    r = sm.OLS(x.gilt, sm.add_constant(x[["ust", "post", "ust_post"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    return {"Test": label, "Beta before": r.params.ust, "Change": r.params.ust_post, "Std. error": r.bse.ust_post,
            "p-value": r.pvalues.ust_post, "Beta after": r.params.ust + r.params.ust_post, "Weeks": len(x)}
b = w.loc["2008":]
t_beta = pd.DataFrame([bt(b, "2008-21 vs 2022-now"),
                       bt(b[~((b.index >= "2022-09-01") & (b.index <= "2022-12-31"))], "Excluding Sep-Dec 2022"),
                       bt(w.loc["2016":], "2016-21 vs 2022-now")]).round(
    {"Beta before": 2, "Change": 2, "Std. error": 2, "p-value": 3, "Beta after": 2})

def events(rate, thr):
    d = rate.resample("W-FRI").last().ffill().diff().reindex(wu.index)
    d = d[d.abs() >= thr]; s = np.sign(d)
    return {"Change weeks": len(d), "US 2Y (bp)": (wu.loc[d.index, "2Y"] * s).mean(),
            "US 10Y (bp)": (wu.loc[d.index, "10Y"] * s).mean(), "UK 2Y (bp)": (wk.loc[d.index, "2Y"] * s).mean(),
            "UK 10Y (bp)": (wk.loc[d.index, "10Y"] * s).mean()}
t1 = fred.get_series("DFEDTAR", observation_start="2001-01-01").dropna()
t2 = fred.get_series("DFEDTARU", observation_start="2008-12-01").dropna()
ev, bank_note = {"Fed funds target": events(pd.concat([t1.loc[:"2008-12-15"], t2]).sort_index(), 0.2)}, ""
try:
    url = ("https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes"
           "&Datefrom=01/Jan/2001&Dateto=now&SeriesCodes=IUDBEDR&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N")
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60); r.raise_for_status()
    br = pd.read_csv(io.StringIO(r.text))
    ev["Bank Rate"] = events(pd.Series(br.iloc[:, 1].values, index=pd.to_datetime(br.iloc[:, 0], format="%d %b %Y")), 0.1)
except Exception as e:
    bank_note = " The Bank Rate download failed, so only Fed weeks are shown."
t_ev = pd.DataFrame(ev).T.round(1); t_ev["Change weeks"] = t_ev["Change weeks"].astype(int)
t_ev = t_ev.rename_axis("Rate").reset_index()

# ---------- section ----------
path, out = "yield_curve_lab.html", "yield_curve_lab_uk.html"
standalone = not os.path.exists(path)
fh = lambda f, first=False: f.to_html(full_html=False, include_plotlyjs=(first and standalone), config=CFG)
tbl = lambda d: d.to_html(index=False, border=0, classes="tbl")
sub = lambda s: f'<p class="sub">{s}</p>'
sec = (
    '<section id="uk"><div class="sh"><span class="num">08</span><div><h2>UK Comparison</h2>'
    + sub("Bank of England fitted nominal gilt spot curves against US Treasuries, using weekly changes because the "
          "London and New York closes are hours apart and daily data understates co-movement. BoE yields are "
          "continuously compounded zero-coupon rates and FRED series are par yields, so levels differ slightly; "
          "changes are comparable.") + '</div></div>'
    + '<h3>Co-movement over time</h3>'
    + sub(f"Rolling 52-week correlation of weekly 10-year yield changes. Full-sample correlation {full:.2f}; "
          f"rolling range {rc.min():.2f} to {rc.max():.2f}.") + fh(f1, True)
    + '<h3>Gilt-minus-Treasury 10-year gap</h3>'
    + sub("UK 10y spot minus US 10y par yield. Compounding explains roughly 0.1 pp of the average level, so the swings "
          "matter more than the level. Dotted lines mark 5 Mar 2009 (BoE announces QE), 23 Jun 2016 (Brexit "
          "referendum) and 23 Sep 2022 (mini-budget).") + fh(f2)
    + '<h3>Factor structure</h3>'
    + sub(f"PCA on daily yield changes, 1Y to 20Y. Variance explained (PC1 / PC2 / PC3): US {vu[0]:.0f} / {vu[1]:.0f} / "
          f"{vu[2]:.0f}%, UK {vk[0]:.0f} / {vk[1]:.0f} / {vk[2]:.0f}%. Weekly correlation of UK and US component "
          f"scores: level {cs[0]:.2f}, slope {cs[1]:.2f}, curvature {cs[2]:.2f}.") + fh(f3)
    + '<h3>Co-movement by maturity</h3>'
    + sub("Correlation of weekly yield changes, UK vs US. Lower correlations at short maturities are consistent "
          "with the front end following domestic policy.") + tbl(t_corr)
    + '<h3>Sensitivity to US moves</h3>'
    + sub("Regression of weekly 10y gilt changes on US changes with a post-2022 interaction, HAC standard errors. "
          "Correlation is similar across periods, so a higher beta reflects relatively larger gilt moves, not a "
          "tighter link.") + tbl(t_beta)
    + '<h3>Policy-rate change weeks</h3>'
    + sub("Mean weekly yield move in weeks when the policy rate changed, signed so a rate rise is positive. "
          "Descriptive only: most moves are priced in before decisions, so this shows co-movement around "
          "decisions, not causal effects." + bank_note) + tbl(t_ev)
    + '<p class="note">Sample July 2001 to September 2026. BoE data are subject to revision; downloaded October 2026.</p></section>')

css = ("body{font-family:Inter,'Segoe UI',Arial,sans-serif;background:#f3f4f6;color:#1f2937;margin:0}"
       "main{max-width:1240px;margin:0 auto;padding:24px}section{background:#fff;border-radius:12px;padding:26px 30px;"
       "box-shadow:0 1px 3px rgba(0,0,0,.1)}.sh{display:flex;gap:16px;margin-bottom:14px;border-bottom:1px solid #e5e7eb;"
       "padding-bottom:14px}.num{font-weight:700;color:#fff;background:#1d4ed8;border-radius:8px;padding:6px 10px;"
       "height:fit-content}h2{margin:0}.sub{color:#4b5563;line-height:1.55;font-size:15px}h3{margin:26px 0 8px}"
       ".tbl{border-collapse:collapse;width:100%}.tbl th{background:#f3f4f6;text-align:left;padding:10px 12px}"
       ".tbl td{padding:9px 12px;border-top:1px solid #e5e7eb}.note{font-size:14px;color:#6b7280}")
if standalone:
    html = f"<!doctype html><html><head><meta charset='utf-8'><title>UK comparison</title><style>{css}</style></head><body><main>{sec}</main></body></html>"
else:
    html = open(path, encoding="utf-8").read()
    html = html.replace('<a href="#heat">Heatmap</a></nav>', '<a href="#heat">Heatmap</a><a href="#uk">UK comparison</a></nav>')
    html = html.replace("</main>", sec + "</main>", 1)
    html = html.replace("for US Treasuries, July 2001", "for US Treasuries with a UK gilt comparison, July 2001")
    html = html.replace("Educational project; not investment advice.",
                        "UK data: Bank of England fitted nominal gilt spot curves; Fed funds target from FRED; Bank Rate from the "
                        "Bank of England database. Educational project; not investment advice.")
open(out, "w", encoding="utf-8").write(html)
print("Saved", out, "| standalone page" if standalone else "| inserted into existing report", "| %.1f MB" % (len(html) / 1e6))
files.download(out)

In [ ]:
#8 TEXT FILES
import re
from google.colab import files
h = open("yield_curve_lab_uk.html", encoding="utf-8").read()
R = [
 ("Observed yield", "Bootstrapped zero yield"),
 ("Allowed range", "Search range"),
 ("Six-parameter Nelson-Siegel-Svensson curves fitted by least squares to the 11 constant-maturity Treasury yields each day, warm-started from the previous day's solution.",
  "Nelson-Siegel-Svensson curves fitted each day to zero-coupon yields bootstrapped from the 11 constant-maturity Treasury par yields."),
 ("A parameter sits at its optimiser bound on about a quarter of days with no loss of fit, so the analysis relies on fitted yields rather than raw parameters. Fit errors rise in 2001-02, 2008-09 and 2022-23.",
  "Lambdas are chosen by grid search each day and smoothed with a 21-day rolling median, and the betas then come from OLS, so no parameter sits at an optimiser bound. Mean fit error is about 4 bp. Against the Fed's published zero-coupon curve the fit differs by about 12 bp at 10Y and 14 bp at 30Y (RMSE), partly because Treasury CMT yields themselves differ from the Fed's par curve by a similar amount."),
 ("Covariance PCA on daily yield changes. Excluding", "Covariance PCA on daily changes in zero-coupon yields. Excluding"),
 ("Method: daily Nelson-Siegel-Svensson least-squares fits, covariance PCA",
  "Method: par yields bootstrapped to zero-coupon, daily Nelson-Siegel-Svensson fits (grid-searched, smoothed lambdas, OLS betas) benchmarked against the Federal Reserve's Gürkaynak-Sack-Wright curve, covariance PCA"),
 ("BoE yields are continuously compounded zero-coupon rates and FRED series are par yields, so levels differ slightly; changes are comparable.",
  "Both sides are continuously compounded zero-coupon yields: the Bank of England curve and the Federal Reserve's Gürkaynak-Sack-Wright Treasury curve."),
 ("UK 10y spot minus US 10y par yield. Compounding explains roughly 0.1 pp of the average level, so the swings matter more than the level.",
  "UK 10y zero-coupon yield minus US 10y zero-coupon yield, on the same compounding basis, so the average gap (about -0.2 pp) is a like-for-like difference."),
 ("not a tighter link.", "not a tighter link. The rise is significant against 2008-21 (p about 0.03) but not against 2016-21 (p about 0.12), so treat it as suggestive."),
]
for old, new in R:
    if old not in h: print("NOT FOUND:", old[:60])
    h = h.replace(old, new)
for pat in (r" A small penalty in the fit stops.*?offsetting each other\.", r" At an optimiser bound on [\d,]+ of [\d,]+ days\."):
    h, n = re.subn(pat, "", h)
    if n == 0: print("NOT FOUND:", pat[:40])
open("yield_curve_lab_final.html", "w", encoding="utf-8").write(h)
files.download("yield_curve_lab_final.html")

In [ ]:

#9 five fixes, writes yield_curve_lab_report.html
import re, json, base64, numpy as np
from google.colab import files
h = open("yield_curve_lab_final.html", encoding="utf-8").read()

h = h.replace("&lt;NA&gt;", "n/a")
h = h.replace("<td>2001-07</td>", "<td>2001–2007</td>")
for a, b in [("2008-15", "2008–2015"), ("2008-21", "2008–2021"), ("2016-21", "2016–2021"), ("2022-now", "2022–now")]:
    h = h.replace(a, b)
for a, b in [
    ("10y-3m recession AUC", "10y-3m AUC (in-sample)"),
    ("so no parameter sits at an optimiser bound.", "so no parameter sits at an optimiser bound. Betas can still be large and offsetting, so the beta panel is clipped to its 1st-99th percentile and a few extreme days fall outside it."),
    ("Educational project; not investment advice.", "US zero-coupon yields in the UK comparison come from the Federal Reserve Board's Gürkaynak-Sack-Wright dataset (a staff research product, not an official release). Educational project; not investment advice."),
]:
    if a not in h: print("NOT FOUND:", a[:40])
    h = h.replace(a, b)

def bal(s, p, a, b):
    d = 0
    for k in range(p, len(s)):
        d += (s[k] == a) - (s[k] == b)
        if d == 0: return k
arr = lambda y: (np.frombuffer(base64.b64decode(y["bdata"]), dtype=np.dtype(y["dtype"])).astype(float)
                 if isinstance(y, dict) else np.array(y, float))
m = re.search(r'Plotly\.newPlot\(\s*"plot_params"\s*,\s*', h)
p = m.end(); e = bal(h, p, "[", "]")
data = json.loads(h[p:e + 1])
v = np.concatenate([arr(d["y"]) for d in data[:4]])
lo, hi = np.nanpercentile(v, [1, 99]); pad = (hi - lo) * 0.1
new = json.dumps([round(lo - pad, 2), round(hi + pad, 2)])
k = h.index("Percent", e)
j = h.rindex('"yaxis":{', 0, k)
r = re.compile(r'"range":\[[^\]]*\]').search(h, j, bal(h, h.index("{", j), "{", "}"))
h = h[:r.start()] + '"range":' + new + h[r.end():]
print("beta axis range ->", new)

open("yield_curve_lab_report.html", "w", encoding="utf-8").write(h)
files.download("yield_curve_lab_report.html")

In [ ]:
#10 out-of-sample test
import numpy as np, statsmodels.api as sm
from sklearn.metrics import roc_auc_score, brier_score_loss
T0, rng = pd.Timestamp("1995-01-31"), np.random.default_rng(0)

def oos(col):
    d = rec_m[[col, "rec_in_12m"]].dropna(); idx = d.index; X = sm.add_constant(d[col]); y = d["rec_in_12m"]
    rows = []
    for t in idx[idx >= T0]:
        tr = idx[idx <= t - pd.DateOffset(months=12)]            # outcomes already realised at t
        try: p = sm.Probit(y[tr], X.loc[tr]).fit(disp=0).predict(X.loc[[t]]).iloc[0]
        except Exception: p = np.nan
        rows.append((t, p, y[tr].mean(), y[t], d.loc[t, col]))
    return pd.DataFrame(rows, columns=["t", "p", "base", "y", "s"]).set_index("t").dropna()

def ci(df, B=1000, L=12):
    p, y, n, out = df.p.values, df.y.values, len(df), []
    for _ in range(B):
        ix = np.concatenate([np.arange(s, s + L) for s in rng.integers(0, n - L + 1, int(np.ceil(n / L)))])[:n]
        if 0 < y[ix].sum() < n: out.append(roc_auc_score(y[ix], p[ix]))
    return np.percentile(out, [2.5, 97.5])

for col in ["s_10y2y", "s_10y3m"]:
    df = oos(col); lo, hi = ci(df)
    print(f"{col}: test months {len(df)} ({int(df.y.sum())} in recession) | in-sample AUC same months {roc_auc_score(df.y, -df.s):.2f} | "
          f"OUT-OF-SAMPLE AUC {roc_auc_score(df.y, df.p):.2f} (95% CI {lo:.2f}-{hi:.2f}) | "
          f"Brier skill vs base rate {1 - brier_score_loss(df.y, df.p) / brier_score_loss(df.y, df.base):+.2f}")

In [ ]:
#11 adds the test table
from google.colab import files
h = open("yield_curve_lab_report.html", encoding="utf-8").read()
rows = [("10y-2y", 369, 28, "0.71", "0.61", "0.25–0.98", "+0.08"),
        ("10y-3m", 369, 28, "0.74", "0.70", "0.41–0.94", "-0.21")]       # from the out-of-sample cell's output
hd = ["Spread", "Test months", "Recession months", "In-sample AUC (same months)", "Out-of-sample AUC",
      "95% CI (block bootstrap)", "Brier skill vs base rate"]
tbl = ("<table class='tbl'><thead><tr>" + "".join(f"<th>{c}</th>" for c in hd) + "</tr></thead><tbody>"
       + "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows) + "</tbody></table>")
block = ("<h3>Out-of-sample test</h3><p class='sub'>Expanding-window probit from January 1995: each month's forecast uses "
         "only months whose 12-month-ahead outcome was already known. The test window contains three recessions (2001, 2008, 2020).</p>"
         + tbl + "<p class='note'>The full-sample AUCs (0.77 and 0.81) are higher partly because they include the 1990-91 recession. "
         "Out of sample, the 10y-3m ranks recession risk better than chance (0.70), but with three events both intervals include 0.5, "
         "so the evidence is suggestive, not conclusive. Negative Brier skill means the 10y-3m probabilities were worse than simply using "
         "the historical recession frequency, so treat the implied probabilities above as rough. Results use final NBER dates; in real time "
         "the dates arrive months late, so the test is somewhat optimistic.</p>")
key = "<h3>Inversion episodes: 10y-2y</h3>"
assert key in h, "insertion point not found"
open("yield_curve_lab_report_v2.html", "w", encoding="utf-8").write(h.replace(key, block + key, 1))
files.download("yield_curve_lab_report_v2.html")

In [ ]:
#12 hedging demo
import numpy as np, pandas as pd
from sklearn.decomposition import PCA
from google.colab import files

Z = pd.read_csv("zero_yields.csv", index_col=0, parse_dates=True)             # zero yields, %
tau = {"2Y": 2, "5Y": 5, "10Y": 10, "30Y": 30}; cm = {"2Y": "DGS2", "5Y": "DGS5", "10Y": "DGS10", "30Y": "DGS30"}
P = pd.DataFrame({k: 100 * np.exp(-Z[cm[k]] / 100 * t) for k, t in tau.items()})   # constant-maturity zero prices
dP = P.shift(-1) - P                                                             # next-day price change
D = pd.DataFrame({k: t * P[k] * 1e-4 for k, t in tau.items()})                   # DV01 per bond

# PCA loadings from 2001-2011 only
dy = Z.diff().dropna() * 100
L = pd.DataFrame(PCA(3).fit(dy.loc[:"2011"]).components_.T, index=Z.columns, columns=["PC1", "PC2", "PC3"])
l = lambda k, c: L.loc[cm[k], c]
D10 = D["10Y"]
a11, a12, a21, a22 = D["2Y"] * l("2Y", "PC1"), D["30Y"] * l("30Y", "PC1"), D["2Y"] * l("2Y", "PC2"), D["30Y"] * l("30Y", "PC2")
r1, r2 = -D10 * l("10Y", "PC1"), -D10 * l("10Y", "PC2")
det = a11 * a22 - a12 * a21
H = {"Unhedged": {},
     "DV01-neutral, 5Y only": {"5Y": -D10 / D["5Y"]},
     "DV01-neutral barbell, 2Y + 30Y": {"2Y": -0.5 * D10 / D["2Y"], "30Y": -0.5 * D10 / D["30Y"]},
     "PCA hedge (PC1 + PC2 neutral), 2Y + 30Y": {"2Y": (r1 * a22 - a12 * r2) / det, "30Y": (a11 * r2 - r1 * a21) / det}}
s = {n: dP["10Y"] + sum(h[k] * dP[k] for k in h) for n, h in H.items()}
gross = {n: sum(abs(h[k]) * P[k] for k in h) / P["10Y"] for n, h in H.items()}

win = [("2012", "2026"), ("2022", "2023")]
rows = [[n] + ["%.0f%%" % (100 * s[n].loc[a:b].std() / s["Unhedged"].loc[a:b].std()) for a, b in win]
        + ["%.1fx" % gross[n].loc["2012":].mean()] for n in H]
tab = pd.DataFrame(rows, columns=["Strategy", "Residual risk, 2012–2026", "Residual risk, 2022–2023", "Gross hedge / position (avg)"])
print(tab.to_string(index=False))

sec = ("<section id='hedge'><div class='sh'><span class='num'>09</span><div><h2>PCA Hedging Demo</h2>"
       "<p class='sub'>Hedging a long 10-year zero-coupon bond with 2-year and 30-year zeros. PCA loadings are estimated on 2001–2011 "
       "only and tested from 2012 onwards, with hedge ratios recomputed daily from each bond's DV01. Residual risk is the standard "
       "deviation of daily P&amp;L as a percentage of the unhedged position's.</p></div></div>"
       + tab.to_html(index=False, border=0, classes="tbl")
       + "<p class='note'>Assumes constant-maturity zero-coupon bonds on the bootstrapped Treasury zero curve, daily rebalancing, and no "
       "financing, roll-down or transaction costs. The PCA hedge removes exposure to the first two factors (level and slope) and leaves "
       "curvature and idiosyncratic risk. 'Gross hedge / position' is hedge market value relative to the 10-year position, so a "
       "higher number means more leverage for the same risk reduction. Illustrative, not a trading recommendation.</p></section>")
h = open("yield_curve_lab_report_v2.html", encoding="utf-8").read()
h = h.replace("</main>", sec + "</main>", 1).replace('<a href="#uk">UK comparison</a></nav>', '<a href="#uk">UK comparison</a><a href="#hedge">Hedging demo</a></nav>')
open("yield_curve_lab_report_v3.html", "w", encoding="utf-8").write(h)
files.download("yield_curve_lab_report_v3.html")

In [ ]:
#13 hedge check on the FED curve
G = pd.read_csv("gsw_zero.csv", index_col=0, parse_dates=True).loc["2001-07-31":]
cmz = {"2Y": "SVENY02", "5Y": "SVENY05", "10Y": "SVENY10", "30Y": "SVENY30"}
gcols = [f"SVENY{t:02d}" for t in (1, 2, 3, 5, 7, 10, 20, 30)]

def run(Zs, cm, cols):
    Zs = Zs[cols].dropna()
    P = pd.DataFrame({k: 100 * np.exp(-Zs[cm[k]] / 100 * t) for k, t in tau.items()})
    dP = P.shift(-1) - P
    D = pd.DataFrame({k: t * P[k] * 1e-4 for k, t in tau.items()})
    L = pd.DataFrame(PCA(3).fit((Zs.diff().dropna() * 100).loc[:"2011"]).components_.T, index=cols, columns=["PC1", "PC2", "PC3"])
    l = lambda k, c: L.loc[cm[k], c]
    a11, a12, a21, a22 = D["2Y"] * l("2Y", "PC1"), D["30Y"] * l("30Y", "PC1"), D["2Y"] * l("2Y", "PC2"), D["30Y"] * l("30Y", "PC2")
    r1, r2 = -D["10Y"] * l("10Y", "PC1"), -D["10Y"] * l("10Y", "PC2"); det = a11 * a22 - a12 * a21
    H = {"5Y only": {"5Y": -D["10Y"] / D["5Y"]},
         "Barbell 2Y+30Y": {"2Y": -0.5 * D["10Y"] / D["2Y"], "30Y": -0.5 * D["10Y"] / D["30Y"]},
         "PCA hedge 2Y+30Y": {"2Y": (r1 * a22 - a12 * r2) / det, "30Y": (a11 * r2 - r1 * a21) / det}}
    un = dP["10Y"].loc["2012":].std()
    return {n: (dP["10Y"] + sum(h[k] * dP[k] for k in h)).loc["2012":].std() / un * 100 for n, h in H.items()}

print(pd.DataFrame({"Treasury CMT-based zeros": run(Z, {"2Y": "DGS2", "5Y": "DGS5", "10Y": "DGS10", "30Y": "DGS30"}, list(Z.columns)),
                    "Fed GSW zeros": run(G, cmz, gcols)}).round(0).astype(int).astype(str).add("%").to_string())

In [ ]:
#14 hedging result note
from google.colab import files
h = open("yield_curve_lab_report_v3.html", encoding="utf-8").read()
key = "Illustrative, not a trading recommendation.</p></section>"
assert key in h, "insertion point not found"
note = ("<p class='note'><b>Result.</b> Over 2012–2026 a simple 2Y+30Y barbell (37% residual risk) did slightly better than the PCA hedge (39%) "
        "on Treasury CMT-based zeros, while the PCA hedge did best in 2022–2023 (31% against 35%). On the Federal Reserve's smoother GSW zero curve "
        "the PCA hedge leads (32%, against 37% for the barbell and 38% for the 5Y hedge), so noise in the Treasury inputs probably masks part of its "
        "advantage. A fitted curve flatters any hedge, because yields on a smooth model curve are mechanically more correlated than those of the "
        "traded bonds, so real residual risk would be higher than the Fed-based figures. Differences of a few points could arise by chance; "
        "no standard errors were computed.</p>")
h = h.replace(key, "Illustrative, not a trading recommendation.</p>" + note + "</section>", 1)
open("yield_curve_lab_report_v4.html", "w", encoding="utf-8").write(h)
files.download("yield_curve_lab_report_v4.html")

In [ ]:
#15 findings, methods, references, README
from urllib.parse import quote_plus
from google.colab import files
h = open("yield_curve_lab_report_v4.html", encoding="utf-8").read()
A = lambda u, t: f"<a href='{u}' target='_blank' rel='noopener'>{t}</a>"
S = lambda q, t: A("https://scholar.google.com/scholar?q=" + quote_plus(q), t)
ul = lambda items: "<ul style='line-height:1.65;font-size:15px;color:#374151'>" + "".join(f"<li>{i}</li>" for i in items) + "</ul>"
head = lambda n, sid, t, sub: (f"<section id='{sid}'><div class='sh'>" + (f"<span class='num'>{n}</span>" if n else "") +
                               f"<div><h2>{t}</h2><p class='sub'>{sub}</p></div></div>")

findings = head("", "findings", "Key Findings", "Four results, with the caveats that matter.") + ul([
    "<b>Curve fitting.</b> Nelson-Siegel-Svensson curves fitted daily to zero-coupon yields bootstrapped from Treasury par yields fit with a mean error of 4.1 bp (6,293 days, 1,391 above 5 bp). Against the Federal Reserve's published curve the gap is about 12 bp at 10Y and 14 bp at 30Y (RMSE), partly because the Treasury data themselves differ from the Fed's curve.",
    "<b>Recession signal.</b> In-sample, the 10y-3m spread gives an AUC of 0.81 (four recessions since 1983). Out of sample from 1995 it falls to 0.70 (95% CI 0.41–0.94), and the 10y-2y to 0.61 (0.25–0.98), so the evidence is suggestive, not conclusive. Latest spreads: 10y-3m 1.07 pp, 10y-2y 0.46 pp.",
    "<b>UK versus US.</b> Weekly 10-year gilt and Treasury yield changes correlate at 0.76, and the level and slope factors have near-identical structure. Slope moves are less shared (0.45 correlation of weekly factor scores). Gilts' sensitivity to US moves rose from 0.68 to 0.80 after 2022 (p = 0.03 against 2008–2021 but p = 0.12 against 2016–2021), so that rise is suggestive only.",
    "<b>Hedging.</b> Hedging a 10-year zero with 2Y and 30Y bonds over 2012–2026, a PCA hedge left 39% of the unhedged risk, against 37% for a simple barbell and 41% for a 5Y hedge. It did best in 2022–2023 (31% against 35% and 41%). On the Fed's smoother curve the PCA hedge led (32% against 37% and 38%)."]) + "</section>"

methods = head("10", "methods", "Methods and Limitations", "How each result was produced, and where it is weakest.") + (
    "<h3>Methods</h3>" + ul([
    "<b>Data.</b> US Treasury constant-maturity yields (1M–30Y), the NBER recession indicator and the Fed funds target from FRED; Bank of England fitted nominal gilt spot curves and Bank Rate; the Federal Reserve Board's Gürkaynak-Sack-Wright (GSW) curve as an external benchmark and as the US zero-coupon curve in the UK comparison.",
    "<b>Samples.</b> Curve fitting, PCA, UK comparison and hedging: July 2001 to September 2026. Recession probit: 1983 onwards, after the 1981–82 recession. Out-of-sample test: from January 1995.",
    "<b>Curve construction.</b> Par yields are interpolated (PCHIP) onto a semiannual grid and bootstrapped to continuously compounded zero rates; 1M and 3M bills are converted directly. PCHIP was chosen over linear interpolation for its closeness to the Fed's curve at 10Y, 20Y and 30Y, which is mild tuning on the validation data.",
    "<b>NSS fitting.</b> Each day λ1 (0.3–4) and λ2 (4–20) are chosen by grid search and smoothed with a 21-day rolling median; the four betas come from OLS.",
    "<b>PCA.</b> Covariance PCA on daily yield changes in basis points, with component signs normalised.",
    "<b>Recession tests.</b> Probit of a recession 12 months ahead on each spread, with HAC standard errors (12 lags). An inversion needs at least two months of negative monthly-average spread, counts if a recession starts within 24 months, and is excluded if it begins during a recession. The out-of-sample test is an expanding-window probit using only outcomes known at the time, with a block bootstrap (12-month blocks, 1,000 draws) for the AUC interval.",
    "<b>UK comparison.</b> Weekly (Friday) changes to limit the London/New York close-time mismatch; regressions use HAC standard errors (4 lags).",
    "<b>Hedging demo.</b> Constant-maturity zero-coupon bonds, daily rebalancing, PCA loadings estimated on 2001–2011 and tested from 2012, with no costs or financing."])
    + "<h3>Limitations</h3>" + ul([
    "The NSS betas are weakly identified: they can be large and offset each other, so the analysis relies on fitted yields rather than raw parameters.",
    "The fitted curve differs from the Fed's by about 12 bp at 10Y and 14 bp at 30Y (RMSE). Treasury CMT yields themselves differ from the Fed's par curve by 13.5 bp at 10Y, so part of this is data, not method.",
    "There are only four recessions since 1983, three in the out-of-sample window, so confidence intervals are wide. Final NBER dates are used, although in real time they arrive months late. Implied probabilities are rough: the 10y-3m's Brier skill was negative.",
    "The UK sensitivity rise is suggestive, the policy-week table is descriptive with no standard errors, and the three sensitivity tests share data.",
    "BoE data are subject to revision, and the Fed's GSW dataset is a staff research product, not an official release.",
    "The hedging demo is idealised, and fitted curves flatter any hedge."]) + "</section>")

refs = head("11", "refs", "Sources and References", "Data sources link to the providers; paper links open a Google Scholar search for the title.") + (
    "<div class='refs'><h3>Data</h3>" + ul([
    A("https://fred.stlouisfed.org/series/DGS10", "FRED: constant-maturity Treasury yields (DGS1MO to DGS30)") + ", " + A("https://fred.stlouisfed.org/series/USREC", "NBER recession indicator (USREC)") + ", " + A("https://fred.stlouisfed.org/series/DFEDTARU", "Fed funds target upper limit (DFEDTARU)") + ". Federal Reserve Bank of St. Louis.",
    A("https://www.bankofengland.co.uk/statistics/yield-curves", "Bank of England: yield curves") + " (fitted nominal gilt spot curves) and the " + A("https://www.bankofengland.co.uk/boeapps/database/", "BoE statistical database") + " (Bank Rate, series IUDBEDR).",
    "Gürkaynak, Sack and Wright (2006), " + A("https://www.federalreserve.gov/pubs/feds/2006/200628/feds200628.html", "The U.S. Treasury Yield Curve: 1961 to the Present (data)") + " and " + A("https://www.federalreserve.gov/pubs/feds/2006/200628/200628pap.pdf", "paper (PDF)") + ". Federal Reserve Board."])
    + "<h3>Methods</h3>" + ul([
    S("Nelson Siegel 1987 Parsimonious modeling of yield curves", "Nelson and Siegel (1987), Parsimonious modeling of yield curves, Journal of Business"),
    S("Svensson 1994 Estimating and interpreting forward interest rates Sweden 1992-1994", "Svensson (1994), Estimating and interpreting forward interest rates: Sweden 1992–1994, NBER Working Paper 4871"),
    S("Litterman Scheinkman 1991 Common factors affecting bond returns", "Litterman and Scheinkman (1991), Common factors affecting bond returns, Journal of Fixed Income"),
    S("Estrella Mishkin 1996 The yield curve as a predictor of U.S. recessions", "Estrella and Mishkin (1996), The yield curve as a predictor of U.S. recessions, FRBNY Current Issues in Economics and Finance"),
    S("Diebold Li 2006 Forecasting the term structure of government bond yields", "Diebold and Li (2006), Forecasting the term structure of government bond yields, Journal of Econometrics"),
    S("Anderson Sleath 2001 New estimates of the UK real and nominal yield curves", "Anderson and Sleath (2001), New estimates of the UK real and nominal yield curves, Bank of England Working Paper 126")])
    + "</div></section>")

k1, k2, k3 = '<section id="fit"', "</main>", '<a href="#hedge">Hedging demo</a></nav>'
assert k1 in h and k2 in h and k3 in h, "insertion point not found"
h = h.replace(k1, findings + k1, 1).replace(k2, methods + refs + k2, 1)
h = h.replace(k3, '<a href="#hedge">Hedging demo</a><a href="#methods">Methods</a><a href="#refs">References</a></nav>', 1)
h = h.replace("</head>", "<style>.refs a{color:#1d4ed8}.refs li{margin:6px 0}</style></head>", 1)
open("yield_curve_lab_report_v5.html", "w", encoding="utf-8").write(h)

readme = """# Yield Curve Lab

Nelson-Siegel-Svensson (NSS) curve fitting, PCA, recession signals and a UK gilt comparison for US Treasuries, July 2001 to September 2026. Built in Google Colab; the output is a single interactive HTML report.

## Key results
- **Fit:** NSS curves fitted daily to zero-coupon yields bootstrapped from Treasury par yields: mean error 4.1 bp, about 12 bp (10Y) and 14 bp (30Y) RMSE against the Federal Reserve's GSW curve.
- **Recessions:** 10y-3m spread AUC 0.81 in-sample (four recessions since 1983), 0.70 out-of-sample from 1995 (95% CI 0.41-0.94); 10y-2y 0.61 (0.25-0.98). Suggestive, not conclusive.
- **UK vs US:** weekly 10Y correlation 0.76; level and slope factors have similar structure; gilt sensitivity to US moves rose from 0.68 to 0.80 after 2022 (p = 0.03 vs 2008-21, p = 0.12 vs 2016-21).
- **Hedging:** hedging a 10Y zero with 2Y+30Y, a PCA hedge left 39% of unhedged risk (2012-2026) vs 37% for a barbell and 41% for a 5Y hedge.

## Data
FRED (API key needed), Bank of England yield curves and Bank Rate, Federal Reserve Board GSW curve (feds200628). Raw data files are not included; see the report's reference section for links.

## Run it
1. Get a free FRED API key and store it as the Colab secret `FRED_API_KEY`.
2. Download the BoE daily nominal archive and latest ZIPs (links in the report).
3. Run the notebook top to bottom; it writes the HTML report.

## Limitations
Weakly identified NSS betas; fit differs from the Fed's curve; only four recessions; final (not real-time) NBER dates; descriptive policy-week table; idealised hedging demo. Details are in the report.

Educational project, not investment advice.
"""
open("README.md", "w", encoding="utf-8").write(readme)
files.download("yield_curve_lab_report_v5.html"); files.download("README.md")

In [ ]:
#16 Final patch
import re, json
from google.colab import files
h = open("yield_curve_lab_report_v5.html", encoding="utf-8").read()
for a, b in [("6,293 days", "6,295 days"),
             ("2026-09-29 (latest)", "2026-09-29 (recent date)"),
             ("July 2001 to September 2026", "July 2001 to 1 October 2026"),
             ("Journal of Econometrics", "Journal of Econometrics (inspired the OLS-beta step; the model itself is not implemented here)")]:
    print("%-34s %d match(es)" % (a, h.count(a))); h = h.replace(a, b)

def bal(s, p, a, b):
    d = 0
    for k in range(p, len(s)):
        d += (s[k] == a) - (s[k] == b)
        if d == 0: return k
m = re.search(r'Plotly\.newPlot\(\s*"plot_params"\s*,\s*', h)
e = bal(h, m.end(), "[", "]")
j = h.index('"yaxis2":{', e)
r = re.compile(r'"range":\[([^\],]*),([^\]]*)\]').search(h, j, bal(h, h.index("{", j), "{", "}"))
h = h[:r.start()] + '"range":[0,' + r.group(2) + ']' + h[r.end():]
print("lambda axis ->", h[r.start():r.start() + 30])
open("yield_curve_lab_report_v6.html", "w", encoding="utf-8").write(h)
files.download("yield_curve_lab_report_v6.html")

In [ ]:
# 17 README
from google.colab import files
r = open("README.md", encoding="utf-8").read()
edits = [
 ("July 2001 to September 2026", "July 2001 to 1 October 2026"),
 ("the output is a single interactive HTML report.\n", "the output is a single interactive HTML report.\n\n**Live report:** https://edrosling.github.io/yield-curve-lab/\n"),
 ("## Data\n", "## Data (downloaded automatically by the notebook)\n"),
 ("2. Download the BoE daily nominal archive and latest ZIPs (links in the report).\n3. Run the notebook top to bottom; it writes the HTML report.",
  "2. Open the notebook in Google Colab and choose Runtime > Run all (the BoE and Fed data download automatically).\n3. The final report is `yield_curve_lab_report_v6.html`; rename it `index.html` to publish with GitHub Pages."),
 ("## Limitations", "## Files\n- `yield_curve_lab.ipynb` - the notebook (run top to bottom)\n- `index.html` - the final report\n- `requirements.txt` - Python packages\n\n## Reproducibility notes\n- Data refreshes daily, so latest values (such as current spreads) will differ from the report.\n- The out-of-sample and hedging results quoted in the report's text were typed in from the notebook's printed output, so rerunning on new data will not update that text.\n- Intermediate HTML files are written along the way; only the last one is the report.\n\n## Limitations"),
]
for a, b in edits:
    if a not in r: print("NOT FOUND:", a[:40])
    r = r.replace(a, b)
open("README.md", "w", encoding="utf-8").write(r)
files.download("README.md")